In [ ]:
import sys
import os
import pandas as pd
import polars as pl

current_dir = os.getcwd()
src_dir = os.path.dirname(current_dir)
sys.path.append(src_dir)

from data_preprocessing.data_normalization import BMLLNormalizer
from data_preprocessing.data_access import DataAccessFactory
from feature_engineering.order_flow import OrderFlowFeatureEngineering

from pipeline.config import PipelineConfig, DataConfig, ProcessingConfig, StorageConfig, RayConfig, S3Location
from pipeline.pipeline import Pipeline

# Shared configuration
BAR_DURATION_MS = 250

In [ ]:
# Unified Feature Engineering Pipeline (processes both trade and L2Q files)
config_unified = PipelineConfig(
    region='us-east-1',
    data=DataConfig(
        raw_data_path='s3://bmlldata',
        start_date='2024-01-02',
        end_date='2024-01-02',
        exchanges=['ARCX'],
        data_types=['trades', 'level2q']
    ),
    processing=ProcessingConfig(
        # normalization=BMLLNormalizer(),  # Enable to run normalization step
        normalization=None,  # Skip normalization
        feature_engineering=OrderFlowFeatureEngineering(bar_duration_ms=BAR_DURATION_MS)
    ),
    storage=StorageConfig(
        raw_data=S3Location(path='s3://bmlldata'),
        normalized=S3Location(path='s3://orderflowanalysis/intermediate/normalized'),
        features=S3Location(path='s3://orderflowanalysis/intermediate/features'),
        models=S3Location(path='s3://orderflowanalysis/output/models'),
        predictions=S3Location(path='s3://orderflowanalysis/output/predictions'),
        backtest=S3Location(path='s3://orderflowanalysis/output/backtest')
    ),
    ray=RayConfig(runtime_env={"working_dir": src_dir},
                  flat_core_count=3,
                  memory_multiplier=2.0,
                  memory_per_core_gb=4.0,
                  max_retries=3,
                  cpu_buffer=1,
                  file_sort_order="desc")
)

print("Testing Unified Feature Engineering Pipeline...")
pipeline = Pipeline(config_unified)
results = pipeline.run(slice(5))

In [ ]:
# Verify results
if results:
    res_pd = pd.DataFrame(results)
    successful = res_pd.query("message == 'success'").shape[0]
    failed = res_pd.query("message != 'success'").shape[0]
    print(f"Results: {successful} success, {failed} failed")
    
    # Show breakdown by data type
    if 'data_type' in res_pd.columns:
        type_counts = res_pd.groupby(['data_type', 'message']).size().unstack(fill_value=0)
        print("\nBreakdown by data type:")
        print(type_counts)
    
    # Show sample results
    for result in results[:3]:
        if result['message'] == 'success':
            group_key = result.get('group_key', 'N/A')
            print(f"✓ {result.get('data_type', 'unknown')} [{group_key}]: {result.get('row_count', 0):,} features -> {result['output_path']}")
        else:
            print(f"✗ Failed: {result['message'][:100]}")
else:
    print("No results returned")